# 02 - Backtest Iteration

In [ ]:
import osos.chdir(r'C:\Users\avav\Documents\freqtrade')import subprocessimport timeimport zipfileimport jsonimport pandas as pdimport plotly.graph_objects as gofrom plotly.subplots import make_subplotsfrom pathlib import Pathfrom datetime import datetimeRESULTS_DIR = Path(r'C:\Users\avav\Documents\freqtrade\user_data\backtest_results')CONFIG = 'config.json'

In [ ]:
def run_backtest(timerange='20240101-20251231', timeframe='15m', pairs='SOL/USDT:USDT', tag='baseline', extra_args=None):    """Run freqtrade backtest via subprocess, return path to latest result zip."""    args = [        'freqtrade', 'backtesting',        '--config', CONFIG,        '--timerange', timerange,        '--timeframe', timeframe,        '--pairs', pairs,        '--userdir', 'user_data',    ]    if extra_args:        args.extend(extra_args)    print(f"Running: {' '.join(args)}")    t0 = time.time()    log_path = f'C:\\Users\\avav\\AppData\\Local\\Temp\\opencode\\bt_{tag}.log'    Path(log_path).parent.mkdir(parents=True, exist_ok=True)    with open(log_path, 'w') as f:        r = subprocess.run(args, capture_output=True, text=True,                          cwd=r'C:\\Users\\avav\\Documents\\freqtrade')        f.write("STDOUT:\n" + r.stdout + "\nSTDERR:\n" + r.stderr)    elapsed = time.time() - t0    print(f"  exit_code: {r.returncode}, elapsed: {elapsed:.1f}s")    if r.returncode != 0:        print("  STDERR (last 1000 chars):")        print(r.stderr[-1000:])        return None    # Find latest result zip    zips = sorted(RESULTS_DIR.glob('backtest-result-*.zip'))    if not zips:        print("  no backtest result found")        return None    latest = zips[-1]    print(f"  result: {latest.name}")    return latest

In [ ]:
def load_backtest(zip_path):    """Load backtest results into dict of dataframes."""    with zipfile.ZipFile(zip_path) as z:        files = z.namelist()        result = {'files': files}        for f in files:            if f.endswith('backtest_result.json'):                result['stats'] = json.loads(z.read(f).decode())            elif f.endswith('-trades.json'):                result['trades'] = pd.read_json(z.read(f).decode())    return result

In [ ]:
def summarize(stats, trades):    """Print key backtest metrics."""    s = stats['strategy']['MultiAgentStrategy']    print(f"Trades:        {s['trades']}")    print(f"Win rate:      {s['wins'] / s['trades'] * 100:.1f}% ({s['wins']}W / {s['losses']}L)")    print(f"Total PnL:     {s['profit_total']:.2f} {s.get('quote_currency', 'USDT')} ({s['profit_total_pct']:.2f}%)")    print(f"Avg duration:  {s['holding_avg']:.1f}s ({s['holding_avg_s']/60:.1f} min)")    print(f"Max drawdown:  {s['max_drawdown']:.2f} ({s['drawdown_high']*100:.2f}%)")    print(f"Sharpe:        {s.get('sharpe', float('nan')):.2f}")    print(f"Profit factor: {s.get('profit_factor', float('nan')):.2f}")    if trades is not None and len(trades) > 0:        avg_long = trades[trades['is_short']==False]['profit_ratio'].mean() if (trades['is_short']==False).any() else 0        avg_short = trades[trades['is_short']==True]['profit_ratio'].mean() if (trades['is_short']==True).any() else 0        n_long = (trades['is_short']==False).sum()        n_short = (trades['is_short']==True).sum()        print(f"\nLONG:  {n_long} trades, avg pnl {avg_long*100:.2f}%")        print(f"SHORT: {n_short} trades, avg pnl {avg_short*100:.2f}%")

In [ ]:
# Run baseline backtest (current state) — uncomment to run# baseline_zip = run_backtest(timerange='20240101-20251231', tag='baseline')# baseline = load_backtest(baseline_zip)# summarize(baseline['stats'], baseline['trades'])

In [ ]:
# Plot equity curve from latest backtestdef plot_equity(stats, title='Equity'):    s = stats['strategy']['MultiAgentStrategy']    daily_pnl = s.get('daily_profit', {})    if daily_pnl:        dates = sorted(daily_pnl.keys())        cum = np.cumsum([daily_profit_pct(daily_pnl[d]) for d in dates])        fig = go.Figure()        fig.add_trace(go.Scatter(x=dates, y=cum*100, mode='lines', name='Cumulative PnL %'))        fig.add_hline(y=0, line_dash='dash', line_color='gray')        fig.update_layout(title=title, template='plotly_dark', yaxis_title='Cumulative PnL (%)')        fig.show()def daily_profit_pct(d):    if isinstance(d, dict):        return d.get('profit_pct', 0)    return float(d) if d is not None else 0# plot_equity(baseline['stats'], title='Baseline equity')

In [ ]:
# Monthly PnL bar chartdef plot_monthly(trades, title='Monthly PnL'):    if trades is None or len(trades) == 0:        print("no trades to plot")        return    t = trades.copy()    t['close_date'] = pd.to_datetime(t['close_date'], unit='ms', utc=True)    t['month'] = t['close_date'].dt.to_period('M').astype(str)    monthly = t.groupby('month')['profit_ratio'].agg(['sum', 'count', lambda x: (x>0).sum()/len(x)*100])    monthly.columns = ['pnl_pct', 'n_trades', 'win_rate']    monthly['pnl_pct'] *= 100    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,                        subplot_titles=('Monthly PnL (%)', 'Trades per month'))    colors = ['green' if v>=0 else 'red' for v in monthly['pnl_pct']]    fig.add_trace(go.Bar(x=monthly.index, y=monthly['pnl_pct'], marker_color=colors, showlegend=False), row=1, col=1)    fig.add_trace(go.Bar(x=monthly.index, y=monthly['n_trades'], marker_color='steelblue', showlegend=False), row=2, col=1)    fig.update_layout(template='plotly_dark', title=title, height=600)    fig.show()# plot_monthly(baseline['trades'])

In [ ]:
# Exit-reason distributiondef plot_exit_reasons(trades, title='Exit reason distribution'):    if trades is None or len(trades) == 0:        print("no trades to plot")        return    if 'exit_reason' in trades.columns:        reasons = trades['exit_reason'].value_counts()        fig = go.Figure(data=[go.Pie(labels=reasons.index, values=reasons.values, hole=0.4)])        fig.update_layout(title=title, template='plotly_dark')        fig.show()    else:        print("'exit_reason' column not present in trades")# plot_exit_reasons(baseline['trades'])

In [ ]:
# Hold time distribution by win/lossdef plot_holdtime(trades, title='Hold time (minutes) by win/loss'):    if trades is None or len(trades) == 0:        return    t = trades.copy()    t['hold_minutes'] = t['trade_duration'] / 60    wins = t[t['profit_ratio'] > 0]['hold_minutes']    losses = t[t['profit_ratio'] <= 0]['hold_minutes']    fig = go.Figure()    fig.add_trace(go.Histogram(x=wins, name='Win', opacity=0.7, marker_color='green'))    fig.add_trace(go.Histogram(x=losses, name='Loss', opacity=0.7, marker_color='red'))    fig.update_layout(barmode='overlay', template='plotly_dark', title=title, xaxis_title='Hold (min)', yaxis_title='count')    fig.show()# plot_holdtime(baseline['trades'])

In [ ]:
Iteration template ready. Copy cells, run, iterate.